In [6]:
import pandas as pd
import numpy as np
import warnings
import sys
from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from statsmodels.tsa.statespace.sarimax import SARIMAX
import os
from datetime import timedelta
import mlflow

warnings.filterwarnings("ignore")
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
sys.path.insert(0, project_root)

sns.set(style='whitegrid')

In [4]:
from src.utils.extract_data import get_data

In [5]:
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI"))
mlflow.set_experiment("all_tickets_v2")
mlflow.autolog(disable=True)

2025/10/17 00:06:47 INFO mlflow.tracking.fluent: Experiment with name 'all_tickets_v2' does not exist. Creating a new experiment.


# preparando os dados

In [11]:
def load_and_prepare(filepath_or_df):

    df = pd.read_csv(filepath_or_df)

    # converter datas
    df['Date received'] = pd.to_datetime(df['Date received'], errors='coerce')
    df = df.dropna(subset=['Date received'])
    df['date'] = df['Date received'].dt.to_period('D').dt.to_timestamp()

    # agregação diária (caso necessário)
    daily = df.groupby('date').size().rename('ticket_count').reset_index()
    daily = daily.sort_values('date').set_index('date').asfreq('D').fillna(0)
    daily.index.name = 'date'
    daily = daily.reset_index()

    # coluna ticket_count para int
    daily['ticket_count'] = daily['ticket_count'].astype(int)

    return daily

In [ ]:
# extraindo e preparando os dados
df = load_and_prepare("../../../data/rows.csv")

# criar features stores que vão ajudar no modelo

In [19]:
def create_time_features(df):
    # cópia do dataframe
    df = df.copy()

    # features de data
    # extrai dia, mês, ano, dia da semana, semana do ano
    df['day'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year
    df['weekday'] = df['date'].dt.weekday  # 0=Mon
    df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)

    # flags
    # criando flags com shifts de 1,2,3,7,14,30 dias
    flag = [1,2,3,7,14,30]
    for l in flag:
        df[f'flag_{l}'] = df['ticket_count'].shift(l)

    # criando janelas para estatísticas móveis
    windows = [3,7,14,30]
    for w in windows:
        df[f'roll_mean_{w}'] = df['ticket_count'].shift(1).rolling(window=w, min_periods=1).mean()
        df[f'roll_std_{w}'] = df['ticket_count'].shift(1).rolling(window=w, min_periods=1).std().fillna(0)

    # diff é a diferença entre o valor atual e o valor de n períodos atrás
    df['diff_1'] = df['ticket_count'].diff(1)

    # pct_change é a variação percentual entre o valor atual e o valor de n períodos atrás
    df['pct_change_1'] = df['ticket_count'].pct_change(1).fillna(0)

    # is_month_start e is_month_end são flags booleanas indicando o início e o fim do mês
    df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
    df['is_month_end'] = df['date'].dt.is_month_end.astype(int)

    # remover linhas com NaN geradas por shift
    df = df.dropna().reset_index(drop=True)
    
    return df

In [20]:
df = create_time_features(df)

In [ ]:
def temporal_train_test_split(df, test_days=90):
    """
    Separa os últimos `test_days` dias como teste.
    """
    # ordenar por data
    df = df.sort_values('date').reset_index(drop=True)
    # separar em treino e teste
    split_date = df['date'].max() - pd.Timedelta(days=test_days)
    train = df[df['date'] <= split_date].reset_index(drop=True)
    test = df[df['date'] > split_date].reset_index(drop=True)
    return train, test

,date,ticket_count,day,month,year,weekday,weekofyear,flag_1,flag_2,flag_3,...,roll_mean_7,roll_std_7,roll_mean_14,roll_std_14,roll_mean_30,roll_std_30,diff_1,pct_change_1,is_month_start,is_month_end
0,2011-12-31,25,31,12,2011,5,52,81.0,84.0,85.0,...,53.142857,36.821319,63.428571,38.552091,83.700000,49.171866,-56.0,-0.691358,0,1
1,2012-01-01,14,1,1,2012,6,52,25.0,81.0,84.0,...,55.142857,34.454940,63.214286,38.771874,80.533333,49.808415,-11.0,-0.440000,1,0
2,2012-01-02,27,2,1,2012,0,1,14.0,25.0,81.0,...,55.714286,33.604138,63.071429,38.962436,76.400000,50.019720,13.0,0.928571,0,0
3,2012-01-03,97,3,1,2012,1,1,27.0,14.0,25.0,...,56.571429,32.633608,58.500000,39.187419,76.433333,49.985297,70.0,2.592593,0,0
4,2012-01-04,106,4,1,2012,2,1,97.0,27.0,14.0,...,59.000000,35.199432,56.857143,36.821518,78.933333,49.035479,9.0,0.092784,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2683,2019-05-06,155,6,5,2019,0,19,71.0,182.0,385.0,...,361.714286,180.696536,534.428571,271.796506,629.600000,301.502742,84.0,1.183099,0,0
2684,2019-05-07,84,7,5,2019,1,19,155.0,71.0,182.0,...,298.142857,159.983779,499.642857,287.670647,620.166667,311.950381,-71.0,-0.458065,0,0
2685,2019-05-08,79,8,5,2019,2,19,84.0,155.0,71.0,...,238.857143,149.710196,438.357143,277.392143,610.100000,324.392201,-5.0,-0.059524,0,0
2686,2019-05-09,50,9,5,2019,3,19,79.0,84.0,155.0,...,195.285714,144.728152,377.642857,253.756882,584.433333,335.122693,-29.0,-0.367089,0,0
